# 14. Pacman with AI

[todo: Write better introduction and goal] The purpose of this practical lesson is for us to learn the inner workings of
NeuralNetwork-based agents for game search, via an interactive and engaging environment.
## 14.1. Part 1: New heuristics

(These have been implemented at `multiAgents.py`, starting at around line ~354).

### 14.1.1. "Approach power capsules" heuristic

We figured that **the agent would benefit from prioritizing going for power capsules** in
order to chase away ghosts and gain some leeway + extra score.

```
# Factor 3: Distancia a la cápsula de poder más cercana
        power_capsules = state.getCapsules()
        if power_capsules:
            min_capsule_distance = min(
                manhattanDistance(pacman_pos, cap_pos) for cap_pos in power_capsules
            )

            # Recompensa por acercarse a una cápsula
            score += 5.0 / (min_capsule_distance + 1)
```

We simply compute the distance to the closest power capsule, then make that inversely
proportional to our heuristic score (the closer to a capsule and the shorter the
distance, the better).

We've conservatively given the capsule heuristic a weight of `5.0`. Large enough to be
more attractive than the plain food heuristic (# factor 1), but not attractive enough to
make the agent reckless and ignore the ghost positions.

However, it was at this point that we thought of a neat addition to this heuristic:

**"If the agent has already consumed a capsule and ghosts are currently under the 'scared'
effect, then further capsules should be strongly discouraged!"**

This matches how humans play the game, avoiding the waste of these useful resources
until they are truly needed:

```
# Factor 3: Distancia a la cápsula de poder más cercana
        power_capsules = state.getCapsules()

        if power_capsules:
            power_active = any(
                g.scaredTimer > 0 for g in ghost_states
            )  # detect power capsule effect
            min_capsule_distance = min(
                manhattanDistance(pacman_pos, cap_pos) for cap_pos in power_capsules
            )

            if not power_active:
                score += 5.0 / (min_capsule_distance + 1)
            else:
                # Strongly discourage consuming capsules while powered
                score -= 100.0 / (min_capsule_distance + 1)
```

We check whether the power capsule effect is active via the query:
``
power_active = any(g.scaredTimer > 0 for g in ghost_states)
``

If it is active, we discourage moving toward the nearest capsule with a weight of
`-100`. We've chosen this weight because it is large enough to be meaningful and
overpower the **'chase ghosts when under the capsule's effect'** heuristic. We deem
preserving power capsules more important than getting a handful of extra points from
eating ghosts.

### 14.1.2. "Avoid undoing moves" heuristic

We also figured that the agent would benefit from avoiding "undoing" moves it has just
done.

For instance: if the agent has just moved "right", it makes sense to discourage it from
moving "left" right afterward, unless there's a good reason to do so (e.g. ghosts are
approaching from the right).

In order to implement this, **the agent obviously needs a way of knowing what its last
action was**. We can easily achieve this via an instance attribute, though we'll also need
to make some small modifications at `multiAgents/NeuralAgent.getAction`, in order to
store the chosen action for the next iteration to use.

We defined the following helper in order to assist with this:

```
def _return_action(self, action):
    self.last_action = action
    return action
```

And modified all relevant `return` instances at `NeuralAgent.getAction()`.

Then, we simply added the following factor at `NeuralAgent.evaluationFunction()`:

```
# Factor 4: Discourage "undoing" moves
        opposites = {
            Directions.NORTH: Directions.SOUTH,
            Directions.SOUTH: Directions.NORTH,
            Directions.EAST: Directions.WEST,
            Directions.WEST: Directions.EAST,
        }

        if self.last_action in opposites:  # (last move might've been STOP)
            undo = opposites[self.last_action]
            if undo in legal_actions:
                score -= 10
```

We chose a weight of 10 because it's big enough to be visually noticeable, without
overpowering the "ghosts are near" and other important penalties.

### 14.1.3. Testing heuristics out

In [1]:
!python pacman.py -p NeuralAgent

Traceback (most recent call last):
  File "/home/pablors/uni/search-algorithms/practice3/pacman.py", line 812, in <module>
    args = readCommand(sys.argv[1:])  # Get game components based on input
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/pablors/uni/search-algorithms/practice3/pacman.py", line 628, in readCommand
    pacmanType = loadAgent(options.pacman, noKeyboard)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/pablors/uni/search-algorithms/practice3/pacman.py", line 704, in loadAgent
    raise Exception('The agent ' + pacman +
Exception: The agent NeuralAgent is not specified in any *Agents.py.
